# Практическая работа 3. Часть 1 — очистка и анализ данных

**Датасет:** IBM HR Analytics Employee Attrition & Performance (Kaggle).
**Объект:** одна строка — один сотрудник компании.
**Целевая переменная:** `Attrition` — уволился сотрудник (Yes) или нет (No).

**План работы**
1. Очистка данных: дубликаты, пропуски, служебные столбцы, логические ошибки.
2. Отбор 25 % строк для разведочного анализа (EDA).
3. Графики, среднее и медиана по выбранным признакам.
4. Формулировка гипотез на 25 % и их проверка на оставшихся 75 %.
5. Вывод.

Выполнил: Гегечкори Георгий

## 1. Загрузка данных и библиотек

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.max_columns", 50)

In [ ]:
import os
FILE = "WA_Fn-UseC_-HR-Employee-Attrition.csv"
# файл ищем рядом с блокнотом или в папке data/
PATH = next((p for p in [FILE, os.path.join("data", FILE)] if os.path.exists(p)), None)

if PATH is None:
    # в Colab файла рядом нет — скачиваем датасет с Kaggle (как в ПР2)
    !pip install -q opendatasets
    import opendatasets as od
    od.download("https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset")
    PATH = "ibm-hr-analytics-attrition-dataset/WA_Fn-UseC_-HR-Employee-Attrition.csv"

df_raw = pd.read_csv(PATH)
df_raw.shape

In [ ]:
df_raw.head()

## 2. Очистка данных

Очистку делаем **до** любого анализа: если в данных есть копии строк, пропуски или невозможные значения,
средние, доли и графики будут посчитаны неверно («Garbage In, Garbage Out»).
Работаем с копией `df`, исходная таблица `df_raw` остаётся нетронутой — так всегда можно сравнить «до» и «после».

In [ ]:
df = df_raw.copy()

### 2.1 Дубликаты

Проверяем три вида повторов:
- **полные копии строк** — одна и та же запись загружена дважды;
- **повтор идентификатора** `EmployeeNumber` — один сотрудник записан несколько раз с разными данными;
- **копии без учёта идентификатора** — один и тот же человек мог попасть в таблицу под разными номерами.

In [ ]:
print("Полных копий строк:", df.duplicated().sum())
print("Повторов EmployeeNumber:", df["EmployeeNumber"].duplicated().sum())
print("Копий без учёта EmployeeNumber:", df.drop(columns="EmployeeNumber").duplicated().sum())

Дубликатов нет ни в одном из трёх смыслов, поэтому удалять нечего. Если бы они нашлись, мы бы оставили первую запись
(`df.drop_duplicates()`): повторная строка давала бы одному сотруднику двойной вес в средних и долях.

### 2.2 Пропуски

In [ ]:
missing = df.isna().sum()
print("Всего пропущенных ячеек:", missing.sum())
missing[missing > 0]

In [ ]:
# пустые строки и строки из одних пробелов pandas не считает пропуском — проверяем отдельно
text_cols = df.select_dtypes(exclude="number").columns
{c: int((df[c].str.strip() == "").sum()) for c in text_cols}

Явных пропусков (NaN) и «скрытых» пропусков (пустых строк) нет. Пропуски появятся ниже — мы сами заменим ими
логически невозможные значения, а заполнять их будем во второй части, **после** разделения на train и test,
чтобы не было утечки данных.

### 2.3 Константные и служебные столбцы

Столбец, в котором одно значение на всю таблицу, не отличает одного сотрудника от другого и ничего не даёт анализу.

In [ ]:
nunique = df.nunique()
nunique[nunique <= 1]

In [ ]:
# прежде чем удалить Over18, проверяем, что он не противоречит возрасту
print("Сотрудников младше 18 лет:", (df["Age"] < 18).sum())
print("Значения Over18:", df["Over18"].unique())

- `EmployeeCount` = 1 и `StandardHours` = 80 у всех — служебные поля выгрузки, удаляем.
- `Over18` = «Y» у всех, и это согласуется с возрастом (младше 18 лет никого нет) — противоречия нет, но информации тоже, удаляем.
- `EmployeeNumber` — идентификатор: он нужен был для проверки дубликатов, но как признак бессмыслен
  (номер не может влиять на увольнение), удаляем.

In [ ]:
df = df.drop(columns=["EmployeeCount", "StandardHours", "Over18", "EmployeeNumber"])
df.shape

### 2.4 Логические ошибки

Логическая ошибка — значение, которое по отдельности выглядит нормально, но противоречит другим полям той же строки
или здравому смыслу. Сформулируем правила, которые обязаны выполняться, и посчитаем нарушения.

In [ ]:
rules = {
    "Возраст от 18 до 65": ~df["Age"].between(18, 65),
    "Начал работать не раньше 14 лет (Age − TotalWorkingYears ≥ 14)": df["Age"] - df["TotalWorkingYears"] < 14,
    "Стаж в компании ≤ общего стажа": df["YearsAtCompany"] > df["TotalWorkingYears"],
    "Стаж в должности ≤ стажа в компании": df["YearsInCurrentRole"] > df["YearsAtCompany"],
    "Лет с повышения ≤ стажа в компании": df["YearsSinceLastPromotion"] > df["YearsAtCompany"],
    "Лет с руководителем ≤ стажа в компании": df["YearsWithCurrManager"] > df["YearsAtCompany"],
    "Доход положительный": df["MonthlyIncome"] <= 0,
    "Шкалы удовлетворённости 1–4": ~df[["EnvironmentSatisfaction", "JobSatisfaction", "RelationshipSatisfaction",
                                       "JobInvolvement", "WorkLifeBalance"]].isin([1, 2, 3, 4]).all(axis=1),
    "Образование 1–5, уровень должности 1–5": ~(df["Education"].between(1, 5) & df["JobLevel"].between(1, 5)),
}
pd.Series({rule: int(mask.sum()) for rule, mask in rules.items()}, name="нарушений")

Базовые правила выполняются. Дополнительно проверим согласованность связанных показателей.

In [ ]:
# оценка эффективности и процент повышения зарплаты должны быть согласованы
df.groupby("PerformanceRating")["PercentSalaryHike"].agg(["min", "max"])

In [ ]:
# более высокий уровень должности должен давать более высокий доход
df.groupby("JobLevel")["MonthlyIncome"].agg(["min", "median", "max"])

Оба соотношения логичны: при оценке 3 повышение 11–19 %, при оценке 4 — 20–25 %; медианный доход растёт с уровнем должности.

#### Ошибка 1: число компаний `NumCompaniesWorked` противоречит стажу

Разница `TotalWorkingYears − YearsAtCompany` — это сколько лет человек проработал **до** текущей компании.
Если она равна 0, значит, текущая компания — первое место работы. Посмотрим, как это сочетается с `NumCompaniesWorked`.

In [ ]:
prev_years = df["TotalWorkingYears"] - df["YearsAtCompany"]
pd.crosstab(df["NumCompaniesWorked"], np.where(prev_years == 0, "только эта компания", "работал раньше"))

**Как понимать поле.** При `NumCompaniesWorked = 1` у 474 из 521 сотрудника стаж до компании равен нулю — значит,
поле считает компании **вместе с текущей** (1 = «работаю только здесь»).

**Что тогда противоречит логике:**
- `NumCompaniesWorked = 0` (197 человек) — так не бывает, текущая компания уже одна. Причём у всех 197 был стаж до неё,
  то есть компаний минимум две;
- `NumCompaniesWorked = 1` при стаже до компании больше нуля (47 человек) — человек работал где-то ещё, значит, компаний не меньше двух.

**Что делаем.** Удалять 244 строки (16,6 % данных) нельзя — остальные поля у этих сотрудников корректны, и мы потеряли бы
много наблюдений, в том числе уволившихся. Угадывать «правильное» число компаний тоже нельзя: известно только, что их не меньше двух.
Поэтому заменяем противоречивые значения на пропуск (NaN): в анализе pandas их просто пропустит, а во второй части
пропуски будут заполнены медианой, посчитанной **только по обучающей выборке**.

In [ ]:
bad_ncw = (df["NumCompaniesWorked"] == 0) | ((df["NumCompaniesWorked"] == 1) & (prev_years > 0))
print("Противоречивых значений NumCompaniesWorked:", bad_ncw.sum())
df.loc[bad_ncw, "NumCompaniesWorked"] = np.nan
df["NumCompaniesWorked"].isna().sum()

#### Ошибка 2: ставки `DailyRate`, `HourlyRate`, `MonthlyRate` не согласуются с доходом

Дневная, часовая и месячная ставки по смыслу должны быть связаны с месячным доходом и друг с другом.

In [ ]:
df[["MonthlyIncome", "DailyRate", "HourlyRate", "MonthlyRate"]].corr().round(3)

In [ ]:
# если бы HourlyRate была настоящей ставкой, то ставка × 160 часов в месяц была бы близка к доходу
ratio = df["HourlyRate"] * 160 / df["MonthlyIncome"]
ratio.describe().round(2)

Корреляция ставок с доходом и между собой — около нуля, а «часовая ставка × 160 ч» отличается от реального дохода
от 0,26 до 13 раз. Значит, эти три столбца не описывают оплату сотрудника: они либо сгенерированы независимо от остальных полей
(датасет синтетический, его создали аналитики IBM), либо означают что-то не указанное в описании.
Использовать признаки с непонятным и противоречивым смыслом нельзя — по ним можно получить ложные выводы. Удаляем все три.

In [ ]:
df = df.drop(columns=["DailyRate", "HourlyRate", "MonthlyRate"])
df.shape

### 2.5 Итог очистки

In [ ]:
print("Было:  ", df_raw.shape)
print("Стало: ", df.shape)
print("Удалено строк:", len(df_raw) - len(df))
print("Удалены столбцы:", sorted(set(df_raw.columns) - set(df.columns)))
print("Пропусков (после замены ошибок):", df.isna().sum().sum())

Все 1470 сотрудников сохранены. Удалено 7 столбцов: 3 константных, идентификатор и 3 противоречивые «ставки».
244 логически невозможных значения `NumCompaniesWorked` заменены пропусками.

Для удобства расчёта долей добавим числовую метку: 1 — уволился, 0 — остался.

In [ ]:
df["left"] = (df["Attrition"] == "Yes").astype(int)
df["left"].value_counts()

## 3. Отбор 25 % данных для разведочного анализа

Берём 25 % строк **случайным образом** (а не подряд): строки в файле могут быть упорядочены, например по отделам или дате
выгрузки, и тогда «первые 25 %» не были бы похожи на остальную компанию. `random_state=42` фиксирует выборку,
чтобы результат воспроизводился.

- `explore` — 25 %: на них смотрим графики и **формулируем** гипотезы;
- `holdout` — оставшиеся 75 %: на них гипотезы **проверяем**. Эти строки мы до проверки не смотрим.

Зачем так: закономерность, найденная при просмотре данных, может оказаться случайной особенностью этой части.
Если она повторяется на отложенных данных — ей можно доверять больше.

In [ ]:
explore = df.sample(frac=0.25, random_state=42)
holdout = df.drop(explore.index)

print("Разведочная часть:", len(explore), "строк")
print("Отложенная часть: ", len(holdout), "строк")
print("Доля уволившихся: explore = {:.1%}, holdout = {:.1%}".format(explore["left"].mean(), holdout["left"].mean()))

Доли уволившихся в частях немного различаются (13 % и 17 %) — это обычный разброс случайной выборки из 368 человек.
Поэтому при проверке будем смотреть не только на направление различий, но и на статистическую значимость.

## 4. Графики по признакам (на 25 %)

Выбор графика зависит от вопроса: долю увольнений в группах сравниваем столбцами, распределение числового признака
в двух группах — boxplot и гистограммой.

### 4.1 Доля увольнений при сверхурочной работе

In [ ]:
rate_ot = explore.groupby("OverTime")["left"].agg(["mean", "count"])
ax = (rate_ot["mean"] * 100).plot(kind="bar", figsize=(6, 4), color=["#4C72B0", "#DD8452"], edgecolor="black")
for i, (r, cnt) in enumerate(zip(rate_ot["mean"], rate_ot["count"])):
    ax.text(i, r * 100 + 0.5, f"{r:.1%}\n(n={cnt})", ha="center")
plt.title("Доля уволившихся по признаку OverTime (explore, 25 %)")
plt.xlabel("Работает сверхурочно")
plt.ylabel("Уволились, %")
plt.ylim(0, 30)
plt.xticks(rotation=0)
plt.show()

Среди тех, кто работает сверхурочно, уволилась каждая пятая (21 %), без переработок — каждый десятый (10 %).
Ось начинается с нуля, а рядом с долей указан размер группы: 94 и 274 человека, — чтобы разница не казалась больше, чем она есть.

### 4.2 Месячный доход уволившихся и оставшихся

In [ ]:
explore.boxplot(column="MonthlyIncome", by="Attrition", figsize=(6, 4))
plt.suptitle("")
plt.title("MonthlyIncome по группам Attrition (explore, 25 %)")
plt.xlabel("Уволился")
plt.ylabel("Месячный доход")
plt.show()

Ящик уволившихся (Yes) расположен ниже: медиана и оба квартиля меньше, чем у оставшихся. В обеих группах много точек сверху —
это высокие доходы руководителей, распределение скошено вправо.

### 4.3 Стаж в компании

In [ ]:
bins = np.arange(0, 42, 2)
plt.figure(figsize=(8, 4))
plt.hist(explore.loc[explore["left"] == 0, "YearsAtCompany"], bins=bins, density=True, alpha=0.6,
         label="Остался", edgecolor="black")
plt.hist(explore.loc[explore["left"] == 1, "YearsAtCompany"], bins=bins, density=True, alpha=0.6,
         label="Уволился", edgecolor="black")
plt.title("Распределение стажа в компании (explore, 25 %)")
plt.xlabel("YearsAtCompany, лет")
plt.ylabel("Доля сотрудников (плотность)")
plt.legend()
plt.show()

Группы разного размера (48 и 320 человек), поэтому строим **плотность** (`density=True`), а не количество — иначе маленькая группа
уволившихся просто терялась бы на фоне большой. У уволившихся заметно больше вес первых двух лет работы.

### 4.4 Доля увольнений по полу

In [ ]:
rate_g = explore.groupby("Gender")["left"].agg(["mean", "count"])
ax = (rate_g["mean"] * 100).plot(kind="bar", figsize=(6, 4), color=["#C44E52", "#4C72B0"], edgecolor="black")
for i, (r, cnt) in enumerate(zip(rate_g["mean"], rate_g["count"])):
    ax.text(i, r * 100 + 0.3, f"{r:.1%}\n(n={cnt})", ha="center")
plt.title("Доля уволившихся по полу (explore, 25 %)")
plt.ylabel("Уволились, %")
plt.ylim(0, 20)
plt.xticks(rotation=0)
plt.show()

Женщины в выборке увольнялись чуть чаще мужчин: 13,9 % против 12,5 %.

## 5. Среднее и медиана

Считаем для признаков с графиков среднее и медиану — по всей разведочной части и отдельно по группам.
Сравнение двух показателей подсказывает форму распределения: если среднее заметно больше медианы, есть «хвост» больших значений.

In [ ]:
num_features = ["MonthlyIncome", "YearsAtCompany", "Age"]

overall = explore[num_features].agg(["mean", "median"]).T
overall.columns = ["среднее", "медиана"]
overall.round(1)

In [ ]:
by_group = explore.groupby("Attrition")[num_features].agg(["mean", "median"]).T.round(1)
by_group

In [ ]:
# для категориального признака аналог среднего — доля уволившихся в группе
pd.DataFrame({
    "доля уволившихся": [explore.loc[explore["OverTime"] == g, "left"].mean() for g in ["No", "Yes"]]
                        + [explore.loc[explore["Gender"] == g, "left"].mean() for g in ["Female", "Male"]],
}, index=["OverTime = No", "OverTime = Yes", "Gender = Female", "Gender = Male"]).round(3)

**Наблюдения:**
- Доход: среднее (≈6850) заметно больше медианы (≈5380) — распределение скошено вправо, небольшая группа с высокой зарплатой
  «тянет» среднее вверх. Поэтому группы по доходу корректнее сравнивать по **медиане**: у уволившихся она ниже
  (≈4640 против ≈5470).
- Стаж в компании: среднее (7,3) больше медианы (6) — тоже правый хвост. У уволившихся и среднее, и медиана меньше.
- Возраст распределён почти симметрично (среднее ≈ медиане ≈ 37); уволившиеся в среднем моложе.

## 6. Гипотезы и их проверка на оставшихся 75 %

Для каждой гипотезы: **наблюдение** на 25 % → **ход мысли** → конкретная **гипотеза** (какие группы и какой показатель сравниваем)
→ что её **опровергнет** → **проверка** на `holdout`.

**Правило подтверждения** (зафиксировано до проверки): различие на отложенных данных в ту же сторону,
что и на разведочных, и статистически значимо (p-value < 0,05).
p-value — вероятность получить такое же или большее различие, если на самом деле разницы между группами нет.
Маленький p-value означает, что различие вряд ли случайно.

- Для долей (уволился / нет в двух группах) — критерий хи-квадрат.
- Для дохода — критерий Манна–Уитни: он сравнивает группы по рангам, а не по средним, поэтому не чувствителен
  к скошенности и высоким зарплатам руководителей.

In [ ]:
def compare_rates(data, mask, name_a, name_b):
    """Сравнивает долю уволившихся в группе A (mask) и группе B (остальные)."""
    a, b = data.loc[mask, "left"], data.loc[~mask, "left"]
    table = pd.crosstab(mask, data["left"])
    p = stats.chi2_contingency(table)[1]
    print(f"{name_a}: {a.mean():.1%} (n={len(a)})")
    print(f"{name_b}: {b.mean():.1%} (n={len(b)})")
    print(f"Разница: {(a.mean() - b.mean()) * 100:+.1f} п.п., отношение: {a.mean() / b.mean():.2f}, p-value = {p:.2g}")
    return a.mean(), b.mean(), p

### Гипотеза 1. Сверхурочная работа и увольнения

**Наблюдение (25 %).** Среди работающих сверхурочно уволились 21 %, без переработок — 10 % (график 4.1).

**Ход мысли.** Переработки — это усталость и нарушенный баланс работы и жизни; логично, что такие сотрудники чаще уходят.
Другое объяснение: сверхурочно работают на определённых должностях (например, в продажах), и уходят именно оттуда.
Но для HR важен сам факт: признак «есть переработки» легко отследить, и он может работать как сигнал риска.

**Гипотеза.** Доля уволившихся среди сотрудников с `OverTime = Yes` выше, чем среди сотрудников с `OverTime = No`.

**Что опровергнет:** доля у работающих сверхурочно такая же или ниже, либо различие незначимо.

In [ ]:
print("Разведочная часть (25 %):")
compare_rates(explore, explore["OverTime"] == "Yes", "OverTime = Yes", "OverTime = No")
print("\nПроверка на отложенной части (75 %):")
h1 = compare_rates(holdout, holdout["OverTime"] == "Yes", "OverTime = Yes", "OverTime = No")

**Результат: гипотеза подтверждена.** На отложенных данных разница даже сильнее: 33 % против 10,5 %,
работающие сверхурочно увольняются примерно в 3 раза чаще; p-value много меньше 0,05.

### Гипотеза 2. Доход уволившихся ниже

**Наблюдение (25 %).** Медианный доход уволившихся ≈4640, оставшихся ≈5470 (график 4.2, раздел 5).

**Ход мысли.** Низкая зарплата — частая причина искать другую работу. Но есть **альтернативное объяснение**:
низкий доход бывает у младших должностей, а младшие сотрудники могут уходить по другим причинам (карьера, учёба).
Тогда дело не в зарплате, а в уровне должности. Чтобы различить объяснения, сравним доход ещё и **внутри одного уровня** —
так же, как в парадоксе Симпсона сравнивают внутри сегментов.

**Гипотеза.** Медианный месячный доход уволившихся ниже, чем у оставшихся; это сохраняется и среди сотрудников одного уровня должности (JobLevel = 1).

**Что опровергнет:** медианы равны или у уволившихся выше; либо внутри уровня разницы нет — тогда общая разница объясняется составом групп.

In [ ]:
def compare_income(data, title):
    left = data.loc[data["left"] == 1, "MonthlyIncome"]
    stay = data.loc[data["left"] == 0, "MonthlyIncome"]
    p = stats.mannwhitneyu(left, stay).pvalue
    print(f"{title}: медиана уволившихся = {left.median():.0f} (n={len(left)}), "
          f"оставшихся = {stay.median():.0f} (n={len(stay)}), p-value = {p:.2g}")
    return left.median(), stay.median(), p

print("Разведочная часть (25 %):")
compare_income(explore, "Все уровни")
compare_income(explore[explore["JobLevel"] == 1], "JobLevel = 1")
print("\nПроверка на отложенной части (75 %):")
h2_all = compare_income(holdout, "Все уровни")
h2_lvl = compare_income(holdout[holdout["JobLevel"] == 1], "JobLevel = 1")

In [ ]:
# доля уволившихся по уровням должности — проверяем альтернативное объяснение
holdout.groupby("JobLevel")["left"].agg(["mean", "count"]).round(3)

**Результат: гипотеза подтверждена, но с важной оговоркой.**
- На отложенных данных разница большая: медиана уволившихся ≈2930 против ≈5070, p-value ≈ 1e-13.
- Значительная часть этой разницы — эффект состава: на 1-м уровне должности увольняются 28 %, на остальных — 2,5–15 %,
  а доход на 1-м уровне самый низкий.
- Но и **внутри** 1-го уровня уволившиеся получают меньше (≈2400 против ≈2710, p ≈ 0,0001). Значит, доход связан
  с увольнением не только через должность.

Обратите внимание: на 25 % внутри уровня 1 разницы не было (p ≈ 0,97) — на маленькой выборке эффект не был виден.
Это пример того, почему проверка на большем объёме данных важна.

### Гипотеза 3. Первые два года в компании

**Наблюдение (25 %).** На гистограмме 4.3 у уволившихся заметно больше вес первых двух лет;
медианный стаж уволившихся меньше, чем у оставшихся.

**Ход мысли.** Новые сотрудники ещё не привязаны к компании: если ожидания не совпали с реальностью, уйти проще в начале.
Порог «2 года» выбран по гистограмме до проверки и не подгоняется под результат.

**Гипотеза.** Доля уволившихся среди сотрудников со стажем в компании ≤ 2 лет выше, чем среди сотрудников со стажем больше 2 лет.

**Что опровергнет:** доля у новичков такая же или ниже, либо различие незначимо.

In [ ]:
print("Разведочная часть (25 %):")
compare_rates(explore, explore["YearsAtCompany"] <= 2, "Стаж ≤ 2 лет", "Стаж > 2 лет")
print("\nПроверка на отложенной части (75 %):")
h3 = compare_rates(holdout, holdout["YearsAtCompany"] <= 2, "Стаж ≤ 2 лет", "Стаж > 2 лет")

**Результат: гипотеза подтверждена.** На отложенных данных новички увольняются в 2,5 раза чаще: 32 % против 12,5 %, p-value много меньше 0,05.

### Гипотеза 4. Женщины увольняются чаще мужчин

**Наблюдение (25 %).** Доля уволившихся среди женщин 13,9 %, среди мужчин 12,5 % (график 4.4).

**Ход мысли.** Различие небольшое — 1,4 п.п. при группах по 144 и 224 человека. Один-два человека меняют долю в группе
почти на процентный пункт, так что разница вполне может быть случайной. Проверим, чтобы не сделать вывод по шуму.

**Гипотеза.** Доля уволившихся среди женщин выше, чем среди мужчин.

**Что опровергнет:** на отложенных данных доля у женщин такая же или ниже, либо различие незначимо.

In [ ]:
print("Разведочная часть (25 %):")
compare_rates(explore, explore["Gender"] == "Female", "Женщины", "Мужчины")
print("\nПроверка на отложенной части (75 %):")
h4 = compare_rates(holdout, holdout["Gender"] == "Female", "Женщины", "Мужчины")

**Результат: гипотеза не подтверждена.** На отложенных данных картина обратная: у женщин 15,1 %, у мужчин 18,5 %,
и это различие тоже незначимо (p > 0,05). То есть различие на 25 % было случайным, а связи пола с увольнением
по этим данным не видно. Это хороший пример, зачем нужна проверка на отложенных данных: без неё мы сделали бы
неверный вывод по маленькой выборке.

## 7. Сводка и вывод

In [ ]:
summary = pd.DataFrame([
    ["1. OverTime = Yes → увольняются чаще", f"{h1[0]:.1%} vs {h1[1]:.1%}", h1[2], "подтверждена"],
    ["2. Доход уволившихся ниже (все уровни)", f"{h2_all[0]:.0f} vs {h2_all[1]:.0f}", h2_all[2], "подтверждена"],
    ["2. … внутри JobLevel = 1", f"{h2_lvl[0]:.0f} vs {h2_lvl[1]:.0f}", h2_lvl[2], "подтверждена"],
    ["3. Стаж ≤ 2 лет → увольняются чаще", f"{h3[0]:.1%} vs {h3[1]:.1%}", h3[2], "подтверждена"],
    ["4. Женщины увольняются чаще", f"{h4[0]:.1%} vs {h4[1]:.1%}", h4[2], "не подтверждена"],
], columns=["Гипотеза", "Результат на 75 % (группа vs остальные)", "p-value", "Вывод"])
summary["p-value"] = summary["p-value"].map(lambda p: f"{p:.2g}")
summary

**Очистка.** Дубликатов и явных пропусков в данных нет. Найдены и обработаны логические ошибки:
244 невозможных значения числа компаний (`NumCompaniesWorked` = 0 при наличии текущей работы и противоречия со стажем)
заменены пропусками; три столбца «ставок» удалены, так как не согласуются с доходом. Удалены также константные столбцы
и идентификатор. Все 1470 сотрудников сохранены.

**Анализ.** На случайных 25 % данных по графикам и средним/медианам сформулированы 4 гипотезы, проверенные на оставшихся 75 %:
- **сверхурочная работа** — самый сильный из найденных сигналов: такие сотрудники увольняются примерно в 3 раза чаще;
- **первые два года** в компании — период повышенного риска: увольняются в 2,5 раза чаще;
- **низкий доход** связан с увольнением; часть этой связи объясняется тем, что чаще уходят сотрудники младших должностей,
  но и при одинаковой должности уволившиеся получали меньше;
- **пол** с увольнением не связан: различие на 25 % оказалось случайным и на 75 % поменяло направление.

**Ограничения.** Найдены связи, а не причины: по срезу без дат нельзя утверждать, что именно переработки *вызывают* увольнение.
Данные синтетические (созданы IBM), поэтому выводы описывают этот набор, а не конкретную реальную компанию.
Для задачи из ПР1 (найти сотрудников с риском ухода) признаки `OverTime`, `YearsAtCompany`, `MonthlyIncome` и `JobLevel`
выглядят наиболее полезными для модели.